In [1]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv("../.env")

CITY_NAME = os.getenv("CITY_NAME", "Hyderabad")
COUNTRY_CODE = os.getenv("COUNTRY_CODE", "PK")
TIMEZONE = os.getenv("TIMEZONE", "Asia/Karachi")

print("City:", CITY_NAME)
print("Country:", COUNTRY_CODE)
print("Timezone:", TIMEZONE)

City: Hyderabad
Country: PK
Timezone: Asia/Karachi


In [2]:
import requests
import pandas as pd

geocoding_url = "https://geocoding-api.open-meteo.com/v1/search"

params = {
    "name": CITY_NAME,
    "count": 10,
    "language": "en",
    "format": "json",
    "country_code": COUNTRY_CODE,
}

response = requests.get(geocoding_url, params=params, timeout=20)
response.raise_for_status()

geo_data = response.json()

locations = geo_data.get("results", [])

if not locations:
    raise ValueError("No location found for Hyderabad, Pakistan.")

locations_df = pd.DataFrame(locations)

locations_df[["name", "country", "admin1", "latitude", "longitude", "timezone"]]

,name,country,admin1,latitude,longitude,timezone
0,Hyderabad,India,Telangana,17.38405,78.45636,Asia/Kolkata
1,Hyderābād,Pakistan,Sindh,25.39689,68.37718,Asia/Karachi
2,Hyderabad,Pakistan,Punjab,31.34602,71.69621,Asia/Karachi
3,Hyderabad,Pakistan,Sindh,25.50000,69.59444,Asia/Karachi
4,Hyderabad,Pakistan,Punjab,29.78445,71.89209,Asia/Karachi
5,Hyderabad,Pakistan,Punjab,33.06676,73.13763,Asia/Karachi
6,Hyderabad City,Pakistan,Sindh,25.38759,68.36326,Asia/Karachi
7,Hyderabad Colony,Pakistan,Punjab,32.14119,72.63568,Asia/Karachi
8,Hyderabad Town,Pakistan,Sindh,25.39815,68.33515,Asia/Karachi
9,Hyderabad Lines,Pakistan,Sindh,24.92360,67.19420,Asia/Karachi


In [3]:
hyderabad = locations_df[
    (locations_df["name"].str.lower() == "hyderabad") &
    (locations_df["country"].str.lower() == "pakistan") &
    (locations_df["admin1"].str.lower() == "sindh")
].iloc[0]

latitude = hyderabad["latitude"]
longitude = hyderabad["longitude"]
timezone = hyderabad.get("timezone", TIMEZONE)

print("Selected city:", hyderabad["name"])
print("Country:", hyderabad["country"])
print("Region:", hyderabad["admin1"])
print("Latitude:", latitude)
print("Longitude:", longitude)
print("Timezone:", timezone)

Selected city: Hyderabad
Country: Pakistan
Region: Sindh
Latitude: 25.5
Longitude: 69.59444
Timezone: Asia/Karachi


In [ ]:
air_quality_url = "https://air-quality-api.open-meteo.com/v1/air-quality"

params = {
    "latitude": latitude,
    "longitude": longitude,
    "current": [
        "us_aqi",
        "pm10",
        "pm2_5",
        "carbon_monoxide",
        "nitrogen_dioxide",
        "sulphur_dioxide",
        "ozone",
        "dust",
        "uv_index",
    ],
    "hourly": [
        "us_aqi",
        "pm10",
        "pm2_5",
        "carbon_monoxide",
        "nitrogen_dioxide",
        "sulphur_dioxide",
        "ozone",
        "dust",
        "uv_index",
    ],
    "timezone": TIMEZONE,
    "forecast_days": 1,
}

response = requests.get(air_quality_url, params=params, timeout=20)
response.raise_for_status()

aq_data = response.json()

aq_data.keys()